# Is my model supported?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/02_is_my_model_supported.ipynb)

TensorEE can run a model in Earth Engine only if every operation in it has a TensorEE building
block, at settings the library can compute exactly. This notebook checks that for a PyTorch
model, and explains the answer in plain words: what is supported, what is refused and why,
and what you can do next.

**It runs entirely on your computer or in Colab.** It needs no Earth Engine account, makes no
Earth Engine request and downloads no weights: every example model is built with random
weights, which is all a support check needs. The check looks at the model's structure, never
at its predictions.

The notebook works through four examples, then has a cell for your own model:

1. a U-Net that is supported;
2. two models that are refused, and what the refusal says;
3. an attention model, which goes through a different path, the converter;
4. your model.

## 1. Install

The `swin` extra brings segmentation-models-pytorch 0.5.0 and timm 1.0.30, the versions the
library was checked with. PyTorch is already installed in Colab.

In [ ]:
%pip install --quiet "tensor-ee[swin]"
# Grouped and depthwise convolutions (ResNeXt and MobileNet encoders, for example) became
# supported after the 0.1.0 release. To check such a model, install from the repository instead:
# %pip install --quiet "tensor-ee[swin] @ git+https://github.com/noelEOS/tensor-ee"

## 2. Settings

This notebook has no live step, so it has no project to set: nothing here contacts Earth
Engine. The only setting is the size of the chip the checks use.

In [ ]:
RUN_LIVE = False                  # this notebook never contacts Earth Engine
INPUT_SHAPE = (1, 2, 64, 64)      # one chip: (1, bands, height, width)

In [ ]:
from collections import Counter

import torch
import segmentation_models_pytorch as smp

from tensor_ee import check_compatibility
from tensor_ee.smp import adapt_smp_model

torch.set_num_threads(2)
torch.manual_seed(0)              # random weights, the same every run


def explain(report):
    """Say in plain words what a compatibility report found."""
    facts = report.to_dict()
    if facts["status"] == "supported":
        storage = facts["weight_storage"]
        print("SUPPORTED: every operation in this model has a TensorEE building block.")
        print(f"  {facts['parameter_count']:,} parameters. After BatchNorm is folded into the "
              f"convolutions, {storage['coefficient_count']:,} numbers would be uploaded as weight assets.")
        print(f"  Output for one chip: {facts['output_shape']} (batch, classes, height, width).")
        return
    print(f"NOT SUPPORTED (status: {facts['status']}).")
    if not facts["evaluation"]["model_eval"]:
        print("  The model is in training mode: call model.eval() first.")
    if not facts["evaluation"]["float32_parameters_and_buffers"]:
        print("  Some weights are not float32: TensorEE computes in float32 at most.")
    if facts["trace"]["status"] != "success":
        print("  The model's code could not be traced:", facts["trace"]["error"])
    reasons = Counter((blocker["code"], blocker["message"]) for blocker in facts["blockers"])
    for (code, message), count in reasons.most_common(8):
        print(f"  {count} x {message}   [{code}]")
    if len(reasons) > 8:
        print(f"  ... and {len(reasons) - 8} more kinds of refusal (report.to_dict()['blockers'] lists them all)")
    if facts["unknowns"]:
        print(f"  {len(facts['unknowns'])} later operations were not checked, because they depend "
              "on the refused ones.")

## 3. A supported model: a U-Net with a ResNet-18 encoder

This is the architecture of the trained water model in notebook 01, built here with random
weights (`encoder_weights=None` means nothing is downloaded).

Two calls do the work:

- `adapt_smp_model` prepares a segmentation-models-pytorch model for TensorEE at a fixed chip
  size. It also runs the original model and the prepared one on one test input and checks that
  they give exactly the same numbers.
- `check_compatibility` walks through every operation of the prepared model and says whether
  TensorEE can compute it.

In [ ]:
unet = smp.Unet("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
prepared = adapt_smp_model(unet, input_shape=INPUT_SHAPE)
print("Prepared model matches the original on a test input:",
      prepared.tensor_ee_smp_metadata["torch_equal"])
explain(check_compatibility(prepared, INPUT_SHAPE))

"Supported" means the library can express this model, operation by operation, as Earth Engine
computations. It is not yet a claim about the numbers. Separately, the library has evidence
for parts of the path: the building blocks agree with PyTorch at the settings they were
validated at, and one trained U-Net with a ResNet-18 encoder, prepared this way, ran in Earth
Engine on a 64 × 64 test input and matched PyTorch to within 1.2e-5 in the logits. Other
encoders and architectures have not been run in Earth Engine as whole models. And a supported
model is not necessarily an accurate one: that depends on its training.

Encoders checked offline in the same way: ResNet-18, 34 and 50 and VGG-11 (with BatchNorm), in
`Unet` and `UnetPlusPlus`.

## 4. Refused models

### A LinkNet: transposed convolutions

LinkNet's decoder upsamples with transposed convolutions (`ConvTranspose2d`). TensorEE has no
building block for them in a CNN, so it refuses the model and names each one.

In [ ]:
linknet = smp.Linknet("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
explain(check_compatibility(adapt_smp_model(linknet, input_shape=INPUT_SHAPE), INPUT_SHAPE))

### DeepLabV3+: dilated convolutions, and more

DeepLabV3+ keeps its last encoder stage at a higher resolution by spacing out the taps of its
convolutions (dilation). The CNN path supports convolutions with stride 1 or 2, no dilation,
and "same" zero padding, so this model is refused. The report also names its other missing
operations: the adaptive average pooling and the bilinear resize of its pyramid pooling module.

In [ ]:
deeplab = smp.DeepLabV3Plus("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
explain(check_compatibility(adapt_smp_model(deeplab, input_shape=INPUT_SHAPE), INPUT_SHAPE))

A refusal is a firm answer: TensorEE never replaces an operation it cannot compute exactly with
an approximation. The message names the first operations that block the model; the
operations after them are not checked. Your options:

- **Change the architecture**, for example a U-Net decoder (nearest-neighbour upsampling and
  convolutions) in place of transposed convolutions, and retrain.
- **Ask for the operation.** [Open an issue](https://github.com/noelEOS/tensor-ee/issues)
  with the printed message and your model's architecture. Adding a building block is a
  library change with its own tests and evidence;
  [CONTRIBUTING](https://github.com/noelEOS/tensor-ee/blob/main/CONTRIBUTING.md) describes it.

## 5. Attention models: the converter

Transformer-style models (SegFormer, Swin) do not go through `adapt_smp_model`. It refuses them
and points to the converter, `tensor_ee.converter.convert`, which recognizes the modules of the
models it was built for: SegFormer-B0 and a UPerNet with a Swin-T encoder, both on
224 × 224 chips.

In [ ]:
from tensor_ee.core.errors import UnsupportedOperation
from tensor_ee.core.grid import Grid
from tensor_ee.converter.convert import convert

segformer = smp.Segformer("mit_b0", encoder_weights=None, in_channels=2, classes=2).eval()
try:
    adapt_smp_model(segformer, input_shape=(1, 2, 224, 224))
except ValueError as refusal:
    print("adapt_smp_model:", refusal)

The converter needs the chip's grid. Converting does not compute anything on imagery: it runs
the model's Python code with placeholder tensors to record its structure. (That also means it
runs the model's code, so convert only models whose source you trust.)

Its report gives one verdict per building block: **inside** the settings that were validated in
Earth Engine, or **outside** them.

In [ ]:
chip = Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000), 224, 224)
b0 = convert(segformer, (1, 2, 224, 224), chip, name="segformer_b0")
print(b0.report.lines()[0])

All 18 blocks of SegFormer-B0 are inside their validated records, and the two "layout
operations" only move pixels onto the model's internal grid and back.

SegFormer-B1 has the same structure at larger widths. It converts, but its blocks are
**outside** their records: the library can express it, and nothing validates it yet. An
outside verdict is reported, not refused.

In [ ]:
segformer_b1 = smp.Segformer("mit_b1", encoder_weights=None, in_channels=2, classes=2).eval()
b1 = convert(segformer_b1, (1, 2, 224, 224), chip, name="segformer_b1")
print(b1.report.lines()[0])
print(next(node for node in b1.report.nodes if node.id == "encoder.block1.0").verdict())

And a refusal: SegFormer-B0 on a 512 × 512 chip. At that size SegFormer's attention reduces
its keys and values to a grid larger than one 7 × 7 window, which needs a building block the
library does not have. The converter says so with a code and the module it stopped at.

In [ ]:
try:
    convert(segformer, (1, 2, 512, 512), Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000), 512, 512))
except UnsupportedOperation as refusal:
    print("Refused:", refusal.code)
    print(str(refusal)[:400])

## 6. Your model

Put your model in `MY_MODEL` and its chip shape in `MY_INPUT_SHAPE`, then run the next two
cells. Build the architecture exactly as you trained it, and load your weights if you have
them (random weights give the same answer). For example:

```python
MY_MODEL = smp.Unet("resnet34", encoder_weights=None, in_channels=4, classes=3)
MY_MODEL.load_state_dict(torch.load("my_checkpoint.pth", map_location="cpu", weights_only=True))
MY_INPUT_SHAPE = (1, 4, 256, 256)
```

The check runs locally. It tries the CNN path first (for a segmentation-models-pytorch model,
or a plain PyTorch CNN), and if that does not support the model, the converter for attention
models.

In [ ]:
MY_MODEL = None                      # your torch.nn.Module
MY_INPUT_SHAPE = (1, 2, 224, 224)    # (1, bands, height, width)

In [ ]:
from tensor_ee.core.errors import UnsupportedOperation


def check_my_model(model, input_shape):
    """Try the CNN path, then, if that does not support the model, the converter."""
    model = model.eval()
    print("CNN path (adapt_smp_model and check_compatibility):")
    try:
        if type(model).__module__.startswith("segmentation_models_pytorch"):
            model_for_check = adapt_smp_model(model, input_shape=input_shape)
        else:
            model_for_check = model      # a plain PyTorch CNN is checked as it is
        report = check_compatibility(model_for_check, input_shape)
        explain(report)
        if report.status == "supported":
            return report
    except (ValueError, RuntimeError, UnsupportedOperation) as refusal:
        print("NOT SUPPORTED:", refusal)

    print()
    print("Attention-model path (the converter):")
    height, width = input_shape[2], input_shape[3]
    try:
        try:
            result = convert(model, input_shape, Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000),
                                                      width, height), name="my_model")
        except UnsupportedOperation as refusal:
            if "source_grid=None" not in str(refusal):
                raise
            result = convert(model, input_shape, name="my_model")   # a Swin encoder starts on its own grid
        print(result.report.lines()[0])
        print("Every block's verdict: print(MY_RESULT.report.text())")
        return result
    except UnsupportedOperation as refusal:
        print("NOT SUPPORTED:", refusal.code, "-", str(refusal)[:600])
    except Exception as failure:      # for example, torch.export could not trace the model
        print("NOT SUPPORTED: the converter could not read this model:", type(failure).__name__, failure)


if MY_MODEL is None:
    print("Put your model in MY_MODEL in the cell above, then run this cell again.")
else:
    MY_RESULT = check_my_model(MY_MODEL, MY_INPUT_SHAPE)

## What next

- **Supported, a CNN.** [Notebook 03](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)
  takes a trained segmentation-models-pytorch U-Net from its checkpoint to Earth Engine: it
  plans the tiles, uploads the weights once, predicts one tile, compares it with PyTorch and
  exports a region.
- **Converted, an attention model.** The
  [converter guide](https://github.com/noelEOS/tensor-ee/blob/main/docs/converter.md) plans the
  model into stages that fit Earth Engine's limits and runs them.
- **Outside the validated records.** The model can be expressed, but its numbers have no
  evidence yet. Running it and comparing it with PyTorch on your own input is how that evidence
  is made; the maintainer can advise.
- **Refused.** See the options under step 4.

The [block catalogue](https://github.com/noelEOS/tensor-ee/blob/main/docs/building-blocks.md)
lists every building block with the settings it was validated at.